# Explore ERP Datasets


In [ ]:
import Pkg

repo_candidates = unique(normpath.([
    pwd(),
    joinpath(pwd(), ".."),
    joinpath(pwd(), "..", ".."),
]))
repo_index = findfirst(path -> isdir(joinpath(path, "datasets")) && isdir(joinpath(path, "src")), repo_candidates)
repo_index === nothing && error("Could not locate repository root from pwd=$(pwd()).")
REPO_ROOT = repo_candidates[repo_index]

Pkg.activate(joinpath(REPO_ROOT, "scripts"))
include(joinpath(REPO_ROOT, "src", "erp_data.jl"))
include(joinpath(REPO_ROOT, "src", "erp_processing.jl"))
include(joinpath(REPO_ROOT, "src", "erp_augmentation.jl"))
include(joinpath(REPO_ROOT, "src", "erp_plot.jl"))


  Activating project at `~/Dokumente/BA2/scripts`
[ Info: Precompiling DataFrames [a93c6f00-e57d-5684-b7b6-d8193f3e46c0] (cache misses: wrong dep version loaded (2), wrong source (4), incompatible header (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling JLD2 [033835bb-8acc-5ee8-8aae-3f567f8a3819] (cache misses: wrong dep version loaded (2), wrong source (4), incompatible header (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling ImageFiltering [6a3955dd-da59-5b1f-98d4-e7296123deb5] (cache misses: wrong dep version loaded (8), incompatible header (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice

In [ ]:
datasets = list_datasets()
dataset_key = "fixations_dataset" in datasets ? "fixations_dataset" : first(datasets)
channels = list_channels(dataset_key)
sort_variables = list_sort_variables(dataset_key)

channel_name = "ch042" in channels ? "ch042" : first(channels)
preferred_sort_variables = ["duration", "rt_ms", "sac_amplitude"]
preferred_available = [name for name in preferred_sort_variables if name in sort_variables]
sort_variable = isempty(preferred_available) ? first(sort_variables) : first(preferred_available)

(dataset_key = dataset_key, channel_name = channel_name, sort_variable = sort_variable)


In [ ]:
events_bundle = load_events(dataset_key)
labels = load_labels(dataset_key)
signal_bundle = load_signal(dataset_key, channel_name)

trial_order = trial_sort_order(events_bundle.events, sort_variable)
sorted_trials = sort_trials(signal_bundle.data_time_trials, trial_order)
zscored_trials = zscore_timepoints(sorted_trials)
erp_image = trials_time_image(zscored_trials)
smoothed_image = smooth_image(erp_image)
resized_image = resize_image(smoothed_image)

(events = size(events_bundle.events), labels = size(labels), signal = size(signal_bundle.data_time_trials), resized = size(resized_image))


In [ ]:
plot_erp_image(dataset_key, channel_name, sort_variable)


In [ ]:
target_trials = min(200, size(signal_bundle.data_time_trials, 2))
augmented = prepare_augmented_images(dataset_key, channel_name, sort_variable; target_trials = target_trials)
(images = length(augmented.images), metadata = size(augmented.metadata))
